# LeafDoctor: generative experiment in one Kaggle run

Thin wrapper around `src/gen/run_experiment.py`; all logic lives in the repo. One run does everything:

1. **Arm B**: `src/gen/cutpaste.py` on the first 15 leaves per class of `data/gen/selection.csv` (405 images).
2. **Arm C**: `src/gen/diffusion_inpaint.py` on the same leaves and placements. SD 1.5 inpainting
   (`stable-diffusion-v1-5/stable-diffusion-inpainting`, CreativeML OpenRAIL-M), fp16, 512x512, 20 steps, on both GPUs.
   Only the background is inpainted (leaf mask shrunk by 8 px), then the original leaf pixels are pasted back.
3. **Train** arms A (no extra data), B (+cut-paste) and C (+diffusion) with the same recipe: `--epochs 5 --lp-epochs 1 --seed 1`.
   Arm A starts right away on GPU 1 while B and C are generated; B and C then train in parallel.
4. **Report**: a pv_test / field_test accuracy and macro-F1 table per arm, every `metrics.json` printed to the log,
   and preview grids in `/kaggle/working/previews/`.

Before running:
1. **Add Data** → attach `ishaanbanga/leafdoctor-data` (mounted read-only at `/kaggle/input/datasets/ishaanbanga/leafdoctor-data/`).
2. **Settings** → Accelerator: **GPU T4 x2**, Internet: on (to clone the repo and download the model, about 2 GB).
3. **Run All**, or *Save Version* to run in the background. Expect roughly 35-40 minutes.
   If time is short, set `MAX_PER_CLASS = 300`, which caps PlantVillage training images per class for all three arms.

In [ ]:
# Parameters
REPO_URL = "https://github.com/bangaishaan/leafdoctor.git"
BRANCH = "main"
DATA_ROOT = "/kaggle/input/datasets/ishaanbanga/leafdoctor-data"   # read-only
WORK = "/kaggle/working"          # synthetic/, results/, previews/ go here
PER_CLASS = 15                    # leaves per class for arms B and C
STEPS = 20                        # diffusion steps
EPOCHS, LP_EPOCHS, SEED = 5, 1, 1
MAX_PER_CLASS = 0                 # 0 = full PlantVillage train set; e.g. 300 if short on time

In [ ]:
%cd /kaggle/working
!rm -rf leafdoctor && git clone --depth 1 --branch {BRANCH} {REPO_URL} leafdoctor
%cd /kaggle/working/leafdoctor
!git log -1 --oneline
!pip install -q -r requirements.txt -r requirements-gen.txt

In [ ]:
import os, torch
for name in ("plantvillage", "plantdoc", "class_map.csv", "split_manifest.csv"):
    assert os.path.exists(os.path.join(DATA_ROOT, name)), f"{name} not found in {DATA_ROOT}; attach the dataset (Add Data)"
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() >= 1, "no GPU: set Accelerator to GPU T4 x2"
if torch.cuda.device_count() < 2:
    print("WARNING: only one GPU; everything will share it and take longer")

In [ ]:
# (b)-(e): generate arms B and C, train A, B and C, print the results table and every metrics.json
!python src/gen/run_experiment.py --data-root "{DATA_ROOT}" --work "{WORK}" --per-class {PER_CLASS} --steps {STEPS} --epochs {EPOCHS} --lp-epochs {LP_EPOCHS} --seed {SEED} --max-per-class {MAX_PER_CLASS}

In [ ]:
# Show the preview grids, then delete the cloned repo so Kaggle has less to save.
import shutil
from IPython.display import Image as ShowImage, display

for p in ("cutpaste_preview.png", "diffusion_preview.png"):
    f = os.path.join(WORK, "previews", p)
    if os.path.exists(f):
        print(f)
        display(ShowImage(f))
os.chdir("/kaggle/working")
shutil.rmtree("/kaggle/working/leafdoctor", ignore_errors=True)
print("deleted /kaggle/working/leafdoctor")
!du -sh /kaggle/working/*